<a href="https://colab.research.google.com/github/sunonmountain/High-Intent-Prospecting-Stack/blob/main/person_finder_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import requests
import pandas as pd
import re
from google.colab import userdata

SERPER_KEY = userdata.get("SERPER_API_KEY")

def sanitize_company_name(raw_name):
    """
    Cleans messy scraped titles (e.g., "Whitecap Consulting Joins Fintech...")
    into searchable brand names (e.g., "Whitecap Consulting").
    """
    # 1. Split by common separators if they exist
    clean_name = re.split(r'\||-|:|–', raw_name)[0].strip()

    # 2. If it's still a full sentence (more than 4 words), aggressively truncate it
    words = clean_name.split()
    if len(words) > 4:
        # Keep only the first 2 words (usually the brand name)
        clean_name = " ".join(words[:2])

    # 3. Remove common corporate suffixes that mess up LinkedIn searches
    clean_name = clean_name.replace("Ltd", "").replace("Limited", "").replace("LLP", "").strip()
    return clean_name

def find_decision_makers(df):
    discovered_people = []

    print(f"🕵️ Elite Discovery: Sourcing leadership for {len(df)} companies...")

    for index, row in df.iterrows():
        raw_company = str(row['company_name'])
        domain = row['domain']

        # Apply the Sanitizer!
        clean_company = sanitize_company_name(raw_company)
        print(f"   🔍 Hunting at: {clean_company} (Raw: {raw_company[:15]}...)")

        # THE EXPANDED X-RAY QUERY (Yields more results, saves credits)
        # We group titles and exclude LinkedIn directory pages
        query = f'site:linkedin.com/in/ ("Managing Director" OR "Founder" OR "CEO" OR "Partner") "{clean_company}" -intitle:"profiles" -inurl:"dir"'

        url = "https://google.serper.dev/search"
        payload = {"q": query, "gl": "gb", "num": 1}
        headers = {'X-API-KEY': SERPER_KEY, 'Content-Type': 'application/json'}

        try:
            response = requests.post(url, json=payload, headers=headers)
            results = response.json().get('organic', [])

            if results:
                hit = results[0]
                # Extract the person's name from the LinkedIn page title
                full_title = hit.get('title')
                name_part = re.split(r'\||-|–', full_title)[0].strip()

                # Split First and Last safely
                name_tokens = name_part.split(' ')
                first_name = name_tokens[0]
                last_name = " ".join(name_tokens[1:]) if len(name_tokens) > 1 else ""

                discovered_people.append({
                    "first_name": first_name,
                    "last_name": last_name,
                    "company": clean_company,
                    "domain": domain,
                    "linkedin_url": hit.get('link')
                })
                print(f"      ✅ Found: {first_name} {last_name}")
            else:
                print(f"      ⚠️ No leadership found for {clean_company}.")

        except Exception as e:
            print(f"      ❌ Error: {e}")

    return pd.DataFrame(discovered_people)

if __name__ == "__main__":
    # Load your B2B list
    harvested_df = pd.read_csv("harvested_B2B_final.csv")

    # Run the upgraded search
    people_df = find_decision_makers(harvested_df)

    # SAVE WITH EXCEL-SAFE ENCODING (utf-8-sig)
    if not people_df.empty:
        os.makedirs('output', exist_ok=True)
        people_df.to_csv("output/discovered_people_V2.csv", index=False, encoding='utf-8-sig')
        print(f"\n🏁 Finished. {len(people_df)} Decision Makers saved with Excel-Safe formatting.")

🕵️ Elite Discovery: Sourcing leadership for 6 companies...
   🔍 Hunting at: Fintech Recruitment Agency (Raw: Fintech Recruit...)
      ✅ Found: Lawrence Wood
   🔍 Hunting at: Penser (Raw: Penser...)
      ✅ Found: Miroslav Sekan
   🔍 Hunting at: Fintech Aura Partners (Raw: Fintech Aura Pa...)
      ✅ Found: Courtney Cardin
   🔍 Hunting at: Moore Kingston (Raw: Moore Kingston ...)
      ✅ Found: Marc Fecher
   🔍 Hunting at: Whitecap Consulting (Raw: Whitecap Consul...)
      ✅ Found: Foluso Laguda
   🔍 Hunting at: Financial Technology Consulting UK (Raw: Financial Techn...)
      ✅ Found: Matt Barrett

🏁 Finished. 6 Decision Makers saved with Excel-Safe formatting.
